# MIHM · Notebook de auditoría y observabilidad para CIMPS2026  
## Estructura DANE/SEN + contenedorización con Podman/Docker

**Objeto auditado:** `MIHM.PDF` / documentación complementaria MIHM v3.0.  
**Propósito:** permitir que un equipo externo observe, reproduzca, audite y emita hallazgos sobre el Modelo de Interacción Homeostática Multicanal (MIHM) sin depender de interpretación informal.

Este notebook está organizado según una adaptación operativa del proceso estadístico DANE/SEN:
1. Necesidad de información
2. Diseño
3. Construcción
4. Recolección / ingesta
5. Procesamiento
6. Análisis
7. Difusión
8. Evaluación

También agrega microservicios reproducibles para separar extracción, validación, cómputo, simulación, trazabilidad y reporte.

## 0. Lectura institucional mínima

Este notebook asume que MIHM no debe presentarse como una promesa estética, sino como una operación estadística/auditable aplicada a una obra sonora y a un vector contextual.

Criterios de auditoría:
- Toda variable debe tener definición, fuente, fórmula o extractor.
- Todo valor emitido debe tener traza.
- Toda ausencia debe conservarse como `null`, no como promedio.
- Toda simulación debe declarar semilla, iteraciones, incertidumbre y límites.
- Toda salida debe poder reconstruirse desde datos, código, configuración y hashes.

In [ ]:
from pathlib import Path
import os, json, hashlib, datetime, re, subprocess, textwrap
import pandas as pd
import numpy as np

ROOT = Path.cwd() / "mihm_cimps2026_audit"
ROOT.mkdir(exist_ok=True)

PDFS = {
    "mihm_paper": Path("/mnt/data/mihm.pdf"),
    "mihm_score": Path("/mnt/data/MIHM_Score.pdf"),
    "systemfriction_mihm": Path("/mnt/data/SystemFriction_MIHM.pdf"),
}

for k, p in PDFS.items():
    print(k, p, "exists=", p.exists(), "sha256=", hashlib.sha256(p.read_bytes()).hexdigest() if p.exists() else None)

## 1. Estructura de carpetas propuesta

La estructura separa evidencia, metadatos, procesamiento, modelos, auditoría y difusión. No se mezcla el documento fuente con resultados derivados.

In [ ]:
folders = [
    "00_gobernanza/acta_necesidad",
    "00_gobernanza/alcance_cimps2026",
    "00_gobernanza/matriz_responsables",
    "01_diseno/metodologia",
    "01_diseno/diccionario_variables",
    "01_diseno/criterios_calidad",
    "02_fuentes/pdf_originales",
    "02_fuentes/audio_original",
    "02_fuentes/contexto_wsv",
    "02_fuentes/metadatos_fuente",
    "03_ingesta/extract_text",
    "03_ingesta/extract_audio",
    "03_ingesta/logs",
    "04_procesamiento/features_audio",
    "04_procesamiento/vector_mihm",
    "04_procesamiento/vector_wsv",
    "04_procesamiento/nti",
    "05_validacion/reglas",
    "05_validacion/tests",
    "05_validacion/hallazgos",
    "06_simulacion/monte_carlo",
    "06_simulacion/escenarios",
    "07_trazabilidad/hashes",
    "07_trazabilidad/ledger",
    "08_reportes/cimps2026",
    "08_reportes/anexos",
    "09_contenedores/services",
    "10_api/contracts",
    "11_dashboard/assets",
]

for f in folders:
    (ROOT / f).mkdir(parents=True, exist_ok=True)

tree = "\\n".join(str(p.relative_to(ROOT)) for p in sorted(ROOT.rglob("*")) if p.is_dir())
print(tree)

## 2. Diccionario operacional de variables MIHM

El equipo auditor debe revisar si cada dimensión está: definida, normalizada, trazable, medible y reproducible.

In [ ]:
variables = pd.DataFrame([
    ["F_s", "Fricción sistémica", 0.100, "Transientes, saturación espectral, rugosidad", "Audio", "Requiere extractor físico"],
    ["G_f", "Gradiente de fricción", 0.100, "Derivada temporal de F_s", "Audio", "Requiere ventanas temporales"],
    ["C_s", "Coherencia sistémica", 0.100, "Continuidad de envolventes/eventos", "Audio", "Validar fórmula exacta"],
    ["R_sem", "Resonancia semántica", 0.100, "Valencia/entropía léxica si hay letra", "Letra/ASR", "Null si no hay señal"],
    ["C_sem", "Campo semántico", 0.100, "Chroma/modo + campo léxico", "Audio/letra", "Null parcial permitido"],
    ["Phi", "Campo cognitivo", 0.100, "Estéreo, profundidad, reverb", "Audio", "Definir medición mid/side"],
    ["I_mc", "Interacción multicanal", 0.083, "Capas activas / solapamiento espectral", "Audio/stems", "Mejor con separación fuente"],
    ["E_r", "Energía relacional", 0.083, "BPM + envolventes + densidad off-beat", "Audio", "No equivale a BPM"],
    ["V_i", "Vector intencional", 0.083, "Dominancia melódica/jerarquía", "Audio/stems", "Null si no hay señal dominante"],
    ["D_i", "Densidad de interacción", 0.075, "Eventos por unidad de tiempo", "Audio/MIDI", "Validar conteo"],
    ["D_cog", "Desfase cognitivo", 0.075, "Complejidad rítmica/métrica", "Audio/MIDI", "Validar sincopación"],
], columns=["variable", "dimension", "peso", "definicion_operativa", "fuente", "criterio_auditoria"])

variables.to_csv(ROOT / "01_diseno/diccionario_variables/mihm_variables.csv", index=False)
variables

## 3. Matriz de calidad estilo DANE/SEN adaptada a MIHM

Esta matriz permite que CIMPS2026 no evalúe solo si el texto es convincente, sino si el sistema cumple condiciones de operación estadística verificable.

In [ ]:
quality = pd.DataFrame([
    ["Relevancia", "La variable responde a una necesidad explícita del modelo", "Cada variable se conecta con IHG, WSV o NTI", "Alta/Media/Baja"],
    ["Exactitud", "El cálculo refleja el fenómeno declarado", "Fórmula, extractor y validación disponibles", "Alta/Media/Baja"],
    ["Oportunidad", "Los datos externos WSV tienen fecha y periodo de referencia", "timestamp, fuente, latencia", "Cumple/No cumple"],
    ["Puntualidad", "La ejecución se puede repetir dentro del periodo declarado", "logs de ejecución", "Cumple/No cumple"],
    ["Accesibilidad", "El auditor puede acceder a insumos y salidas", "rutas, README, API", "Cumple/No cumple"],
    ["Interpretabilidad", "Metadatos suficientes para entender cada salida", "diccionario + explain.weights", "Cumple/No cumple"],
    ["Coherencia", "Variables, pesos y penalizaciones no se contradicen", "tests de consistencia", "Cumple/No cumple"],
    ["Comparabilidad", "Resultados comparables entre obras/ventanas", "normalización [0,1], semillas", "Cumple/No cumple"],
    ["Trazabilidad", "Cada valor tiene fuente/hash/proceso", "ledger append-only", "Cumple/No cumple"],
    ["No simulación indebida", "Ausencia se declara como null", "tests contra imputación silenciosa", "Cumple/No cumple"],
], columns=["criterio", "definicion", "evidencia_requerida", "escala"])

quality.to_csv(ROOT / "01_diseno/criterios_calidad/matriz_calidad_mihm.csv", index=False)
quality

## 4. Ledger mínimo de evidencia

Cada archivo fuente y cada salida derivada debe registrar hash, timestamp, origen, proceso y responsable.

In [ ]:
records = []
for name, path in PDFS.items():
    if path.exists():
        records.append({
            "asset_id": name,
            "path": str(path),
            "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
            "bytes": path.stat().st_size,
            "timestamp_utc": datetime.datetime.utcnow().isoformat(),
            "tipo": "fuente_pdf",
            "estado": "registrado"
        })

ledger = pd.DataFrame(records)
ledger_path = ROOT / "07_trazabilidad/ledger/ledger_assets.csv"
ledger.to_csv(ledger_path, index=False)
ledger

## 5. Reglas de validación MIHM

Estas pruebas no prueban que MIHM sea “verdadero”; prueban que no se contradiga operacionalmente.

In [ ]:
weights = dict(zip(variables["variable"], variables["peso"]))
assert abs(sum(weights.values()) - 1.0) < 0.01, f"Pesos no suman ~1: {sum(weights.values())}"

def clamp01(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    return max(0.0, min(1.0, float(x)))

def validate_vector(v):
    errors = []
    for key in weights:
        if key not in v:
            errors.append(f"Falta variable {key}")
        else:
            val = v[key]
            if val is not None and not (0 <= float(val) <= 1):
                errors.append(f"{key} fuera de rango [0,1]: {val}")
    if v.get("R_sem") is not None and v.get("C_sem") is not None:
        if v["R_sem"] > v["C_sem"]:
            errors.append("R_sem no puede exceder C_sem según restricción MIHM")
    return errors

example_vector = {
    "F_s":0.215, "D_i":0.633, "G_f":0.112, "C_s":0.999, "D_cog":0.004,
    "E_r":0.476, "V_i":0.351, "R_sem":None, "C_sem":None, "I_mc":None, "Phi":None
}

validate_vector(example_vector)

## 6. Cálculo IHG reproducible

La auditoría debe exigir que las penalizaciones estén declaradas y que el límite de penalización sea visible.

In [ ]:
def penalty(A, B, w=0.10, high=0.60, low=0.40):
    if A is None or B is None:
        return 0.0
    return w * A * (1 - B) if A > high and B < low else 0.0

def compute_ihg(v):
    base = sum(weights[k] * float(v[k]) for k in weights if v.get(k) is not None)
    penalties = {
        "Fs_x_Cs": penalty(v.get("F_s"), v.get("C_s")),
        "Di_x_1Er": penalty(v.get("D_i"), v.get("E_r")),
        "Dcog_x_1Rsem": penalty(v.get("D_cog"), v.get("R_sem")),
        "Gf_x_1Vi": penalty(v.get("G_f"), v.get("V_i")),
    }
    total_penalty = min(0.5, sum(penalties.values()))
    return {"base": base, "penalties": penalties, "total_penalty": total_penalty, "IHG_raw": base - total_penalty}

compute_ihg(example_vector)

## 7. Microservicios propuestos

Separación mínima:
- `pdf-extractor`: extrae texto, tablas y referencias.
- `audio-feature-service`: extrae features físicas.
- `mihm-core`: valida vector, pesos, penalizaciones e IHG.
- `wsv-ingestor`: ingesta fuentes externas y declara `null` si falla.
- `montecarlo-service`: ejecuta simulación con semilla e incertidumbre.
- `ledger-service`: hashes, eventos append-only, verificación.
- `report-service`: genera reporte CIMPS2026.
- `dashboard-service`: observabilidad visual.

In [ ]:
compose_yaml = r'''
services:
  pdf-extractor:
    build: ./services/pdf-extractor
    volumes:
      - ../02_fuentes:/data/in:ro
      - ../03_ingesta:/data/out
    command: ["python", "extract_pdf.py"]

  audio-feature-service:
    build: ./services/audio-feature-service
    volumes:
      - ../02_fuentes/audio_original:/audio:ro
      - ../04_procesamiento/features_audio:/features
    command: ["python", "extract_audio_features.py"]

  mihm-core:
    build: ./services/mihm-core
    volumes:
      - ../04_procesamiento:/processing
      - ../05_validacion:/validation
    command: ["python", "validate_and_score.py"]

  wsv-ingestor:
    build: ./services/wsv-ingestor
    environment:
      - NO_SIMULATION=true
    volumes:
      - ../02_fuentes/contexto_wsv:/wsv
    command: ["python", "ingest_wsv.py"]

  montecarlo-service:
    build: ./services/montecarlo-service
    volumes:
      - ../04_procesamiento:/processing:ro
      - ../06_simulacion:/simulation
    command: ["python", "run_montecarlo.py", "--seed", "42", "--n", "50000"]

  ledger-service:
    build: ./services/ledger-service
    volumes:
      - ../:/workspace
      - ../07_trazabilidad:/ledger
    command: ["python", "ledger_verify.py"]

  report-service:
    build: ./services/report-service
    volumes:
      - ../:/workspace
      - ../08_reportes:/reports
    command: ["python", "build_cimps_report.py"]
'''
compose_path = ROOT / "09_contenedores/podman-compose.yml"
compose_path.write_text(compose_yaml.strip(), encoding="utf-8")
print(compose_path)
print(compose_yaml)

## 8. Dockerfile/Containerfile base

Cada microservicio puede tener un `Containerfile` equivalente. Podman lo puede ejecutar sin daemon y con mejor alineación a entornos rootless.

In [ ]:
base_containerfile = r'''
FROM python:3.11-slim

ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1

WORKDIR /app

RUN apt-get update && apt-get install -y --no-install-recommends \
    ffmpeg git build-essential poppler-utils \
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt /app/requirements.txt
RUN pip install --no-cache-dir -r requirements.txt

COPY . /app
'''
for service in ["pdf-extractor","audio-feature-service","mihm-core","wsv-ingestor","montecarlo-service","ledger-service","report-service"]:
    service_dir = ROOT / "09_contenedores/services" / service
    service_dir.mkdir(parents=True, exist_ok=True)
    (service_dir / "Containerfile").write_text(base_containerfile.strip(), encoding="utf-8")
    (service_dir / "requirements.txt").write_text("pandas\nnumpy\npydantic\npytest\n", encoding="utf-8")

print("Containerfiles creados en:", ROOT / "09_contenedores/services")

## 9. Contrato API mínimo

Este contrato permite que el equipo CIMPS2026 audite el sistema como servicio, no solo como PDF.

In [ ]:
openapi = {
  "openapi": "3.1.0",
  "info": {"title": "MIHM Audit API", "version": "0.1.0"},
  "paths": {
    "/health": {"get": {"responses": {"200": {"description": "OK"}}}},
    "/assets/hash": {"post": {"summary": "Registrar hash de evidencia", "responses": {"200": {"description": "Hash registrado"}}}},
    "/mihm/validate": {"post": {"summary": "Validar vector MIHM", "responses": {"200": {"description": "Resultado de validación"}}}},
    "/mihm/score": {"post": {"summary": "Calcular IHG y penalizaciones", "responses": {"200": {"description": "Score calculado"}}}},
    "/wsv/ingest": {"post": {"summary": "Ingestar contexto WSV", "responses": {"200": {"description": "WSV registrado"}}}},
    "/simulation/montecarlo": {"post": {"summary": "Ejecutar simulación Monte Carlo", "responses": {"200": {"description": "Resultados"}}}},
    "/ledger/verify": {"get": {"summary": "Verificar trazabilidad", "responses": {"200": {"description": "Ledger verificado"}}}},
    "/report/cimps2026": {"get": {"summary": "Generar reporte auditoría", "responses": {"200": {"description": "Reporte generado"}}}}
  }
}
api_path = ROOT / "10_api/contracts/openapi_mihm_audit.json"
api_path.write_text(json.dumps(openapi, indent=2, ensure_ascii=False), encoding="utf-8")
print(api_path)

## 10. Checklist para CIMPS2026

Uso del comité:
1. Verificar identidad de documentos y hashes.
2. Confirmar que MIHM declara variables, pesos y fórmula IHG.
3. Confirmar que la ética del silencio se implementa en código.
4. Revisar si las fuentes WSV son reproducibles y fechadas.
5. Ejecutar pruebas contra imputación silenciosa.
6. Ejecutar Monte Carlo con semilla fija.
7. Comparar reporte generado contra documento original.
8. Emitir hallazgos: aceptado, aceptado con observaciones, no reproducible, insuficiente trazabilidad.

In [ ]:
checklist = pd.DataFrame([
    ["Documentos fuente registrados con SHA-256", "ledger_assets.csv", "Pendiente revisión humana"],
    ["Variables MIHM definidas", "mihm_variables.csv", "Listo"],
    ["Pesos suman aproximadamente 1", "assert notebook", "Listo"],
    ["Valores fuera de rango detectados", "validate_vector()", "Listo"],
    ["R_sem <= C_sem cuando ambos existen", "validate_vector()", "Listo"],
    ["Null preservado sin imputación", "tests requeridos", "Pendiente implementación completa"],
    ["IHG reproducible", "compute_ihg()", "Prototipo"],
    ["Monte Carlo con seed", "compose service", "Pendiente motor real"],
    ["API auditable", "openapi_mihm_audit.json", "Borrador"],
    ["Reporte final", "report-service", "Pendiente"],
], columns=["criterio", "evidencia", "estado"])

checklist.to_csv(ROOT / "05_validacion/hallazgos/checklist_cimps2026.csv", index=False)
checklist

## 11. Dictamen inicial del notebook

Estado observado:
- El marco MIHM ya contiene una estructura formal suficiente para ser auditada como especificación.
- El siguiente salto no es escribir más teoría; es separar evidencia, extractores, reglas y resultados.
- La auditoría debe concentrarse en reproducibilidad, trazabilidad, no imputación y estabilidad de las simulaciones.
- Si una variable no tiene extractor verificable, debe quedar marcada como `pendiente`, no como valor simulado.

Resultado operativo: este notebook crea el esqueleto mínimo para pasar de PDF conceptual-operativo a paquete auditable.